# 🧠 Vision-Language Joint Framework for Multimodal Scene Understanding
### Visual Genome • Vision Transformer (ViT) • BLIP • CLIP • Scene Graphs
---
**Student:** AYANSH VERMA (2403A54025)  
**Course:** 24BTCAI CSB18  
**Environment:** Google Colab (Free GPU / T4 Accelerated)

This notebook allows anyone to run and present the complete Multimodal AI pipeline either **directly inside the notebook cells** or as a **live interactive web dashboard**.

## ⚡ Step 1: Verify Hardware Acceleration

In [ ]:
# Ensure GPU is enabled: Runtime -> Change runtime type -> T4 GPU
!nvidia-smi

## 📦 Step 2: Unzip Project Code
*(Upload `vision_language_framework_colab.zip` to the Colab files panel on the left, then run this cell)*

In [ ]:
import os

# Create project directory and unzip
!mkdir -p /content/vision-language-framework
if os.path.exists('/content/vision_language_framework_colab.zip'):
    !unzip -q -o /content/vision_language_framework_colab.zip -d /content/vision-language-framework
    print("✅ Project unzipped successfully!")
else:
    print("⚠️ Please upload 'vision_language_framework_colab.zip' to the left files pane in Colab.")

%cd /content/vision-language-framework
!ls -la

## 🛠️ Step 3: Install Required Libraries

In [ ]:
!pip install -q transformers accelerate streamlit pyngrok networkx timm pillow matplotlib pandas
print("✅ All dependencies installed!")

## 🧪 Step 4: Run Automated Pipeline Tests (GPU Verified)

In [ ]:
!python tests/run_tests.py

## 🎯 Step 5: Live In-Notebook Presentation & Demonstration
Run this cell to process a sample scene and display all visual outputs directly inside Colab (captions, object detections, attributes, and scene graphs) without needing any web tunnel!

In [ ]:
import sys
import os
from PIL import Image
import matplotlib.pyplot as plt
import pandas as pd

# Setup paths
if '/content/vision-language-framework' not in sys.path:
    sys.path.insert(0, '/content/vision-language-framework')

from src.inference import MultimodalInferenceEngine
from src.data_loader import build_synthetic_catalog

# 1. Initialize Engine on GPU
print("⏳ Initializing Multimodal Inference Engine...")
engine = MultimodalInferenceEngine()

# 2. Load Sample Scene
catalog = build_synthetic_catalog()
sample_key = next(iter(catalog.keys()))
sample_meta = catalog[sample_key]
image = Image.open(sample_meta['image_path']).convert('RGB')

# 3. Run Joint Multimodal Inference
question = "What is in the scene?"
print(f"🚀 Processing scene: {sample_meta['title']}...")
results = engine.process(image, question=question)

# 4. Display Captions & VQA Results
print("\n=======================================================")
print("📝 GENERATED IMAGE DESCRIPTIONS & CAPTIONS")
print("=======================================================")
print(f"✨ SOTA BLIP Caption:   \"{results['blip_caption']['caption']}\" (Latency: {results['blip_caption']['latency_s']}s)")
print(f"🏛️ ViT-GPT2 Baseline:  \"{results['baseline_caption']['caption']}\" (Latency: {results['baseline_caption']['latency_s']}s)")
print(f"❓ VQA Answer to '{question}': {results['vqa'].get('answer')}")
print("=======================================================\n")

# 5. Display Visual Detections & Scene Graph Side-by-Side
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
axes[0].imshow(results['annotated_image'])
axes[0].set_title("Object Detections + Attributes (YOLOS + CLIP)", fontsize=14, fontweight='bold')
axes[0].axis('off')

if 'scene_graph_image' in results and results['scene_graph_image']:
    axes[1].imshow(results['scene_graph_image'])
axes[1].set_title("Semantic Scene Graph (NetworkX)", fontsize=14, fontweight='bold')
axes[1].axis('off')

plt.tight_layout()
plt.show()

# 6. Display Detected Entities & Attributes Table
if results['enhanced_detections']:
    table_data = []
    for d in results['enhanced_detections']:
        attrs = ", ".join([a['attribute'] for a in d.get('attributes', [])])
        table_data.append({
            "Detected Object": d['label'],
            "Confidence": f"{d['score']:.2f}",
            "Bounding Box": str(d['box']),
            "Top Predicted Attributes": attrs
        })
    print("\n🔍 DETECTED OBJECTS & PREDICTED ATTRIBUTES:")
    display(pd.DataFrame(table_data))

# 7. Display Scene Graph Triplets
if results['triplets']:
    print("\n🕸️ RELATIONSHIP TRIPLETS (Subject - Predicate - Object):")
    triplet_df = pd.DataFrame([{
        "Subject": t['subject'],
        "Predicate": t['predicate'],
        "Object": t['object'],
        "Confidence": f"{t.get('score', 0):.2f}"
    } for t in results['triplets']])
    display(triplet_df)

## 🌐 Step 6: Launch Full Interactive Streamlit Web App
Want to give your audience the complete live interactive web dashboard? Run this cell to start Streamlit with a public URL tunnel.

In [ ]:
# 1. Install localtunnel for public URL exposure
!npm install -g localtunnel

# 2. Launch Streamlit server in the background
!streamlit run frontend/app.py --server.port 8501 &

# 3. Print your LocalTunnel password
import urllib.request
endpoint_ip = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip()
print("=================================================================")
print(f"🔑 YOUR TUNNEL PASSWORD / ENDPOINT IP:  {endpoint_ip}")
print("Click the link below and paste this IP when prompted to open the app!")
print("=================================================================\n")

# 4. Expose the port
!npx localtunnel --port 8501